# template_overhead_check
Settles the chat-template token overhead used by the soft-prompt length figures.

`prompting_arm_fewshot_*` measured overhead 25 through `pipe._apply_chat_template`, the
INFERENCE path. The 888-token cap is a TRAINING-side limit (`maximum_text_tokens` in
`OmniGen2Collator`), so the figures 288 / 956 / 1,535 / 96.23% depend on the training template
carrying the same overhead. This notebook reads both templates from the pinned commit, measures
the overhead on every training row, and recomputes the four figures.

CPU only. No model weights, no GPU, no API calls.

## 1. Repo at the pinned commit

In [1]:
import os, sys, subprocess, json, re, inspect, collections

REPO   = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "50", "origin", COMMIT], check=False)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=False)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("HEAD    :", head)
print("pinned  :", COMMIT)
print("match   :", head == COMMIT)
sys.path.insert(0, REPO)

HEAD    : 18e6f9d5271b517fcb32e999f10df943ae9b8f20
pinned  : 18e6f9d5271b517fcb32e999f10df943ae9b8f20
match   : True


## 2. Drive, splits, tokenizer

In [2]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("OmniGen2/OmniGen2", subfolder="processor")

splits = {s: [json.loads(l) for l in open(f"{DRIVE}/{s}.jsonl", encoding="utf-8")]
          for s in ("train", "val", "test")}
excl  = {l.strip() for l in open(f"{DRIVE}/screen_exclude_from_training.txt") if l.strip()}
used  = [r for r in splits["train"] if r["output_image"] not in excl]

print("tokenizer :", type(tok).__name__)
print("train     :", len(splits["train"]), "rows |", len(used), "after the screen exclusion")
print("val/test  :", len(splits["val"]), "/", len(splits["test"]))

Mounted at /content/drive


tokenizer_config.json:   0%|          | 0.00/8.01k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

processor/tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

processor/tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer : Qwen2Tokenizer
train     : 39893 rows | 39591 after the screen exclusion
val/test  : 98 / 488


## 3. Every chat-template and system-prompt site in the repo
Locates the constants both paths use.

In [3]:
PAT = (r"im_start|apply_chat_template|SYSTEM_PROMPT|system_prompt|"
       r"use_chat_template|maximum_text_tokens|add_generation_prompt")
out = subprocess.run(
    f"grep -rnE '{PAT}' {REPO} --include=*.py --include=*.yml --include=*.yaml",
    shell=True, capture_output=True, text=True).stdout
print(out if out.strip() else "no matches")

/content/OmniGen2/options/ft_lora.yml:9:  use_chat_template: true
/content/OmniGen2/options/ft_lora.yml:10:  maximum_text_tokens: 888
/content/OmniGen2/options/ft.yml:9:  use_chat_template: true
/content/OmniGen2/options/ft.yml:10:  maximum_text_tokens: 888
/content/OmniGen2/OmniGen2-RL/reward_server/reward_server.py:32:def apply_chat_template(prompt, num_images: int = 2):
/content/OmniGen2/OmniGen2-RL/reward_server/reward_server.py:36:    template = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n"
/content/OmniGen2/OmniGen2-RL/reward_server/reward_server.py:38:    template += f"{prompt}<|im_end|>\n<|im_start|>assistant\n"
/content/OmniGen2/OmniGen2-RL/options/omnigen2_edit_rl_single_machine_editscore7b.yml:11:  use_chat_template: true
/content/OmniGen2/OmniGen2-RL/options/omnigen2_edit_rl_single_machine_editscore7b.yml:12:  maximum_text_tokens: 888
/content/OmniGen2/OmniGen2-RL/options/omnigen2_edit_rl_4machine_editscore7b_avg4.yml:11:  use_chat_templat

## 4. Source of both templates
Left: the inference path that produced the measured 25. Right: whatever consumes
`use_chat_template` on the training path.

In [4]:
def show(pattern, ctx=14, root=f"{REPO}/omnigen2"):
    files = subprocess.run(f"grep -rlE '{pattern}' {root} --include=*.py",
                           shell=True, capture_output=True, text=True).stdout.split()
    for f in files:
        L = open(f, encoding="utf-8").read().split("\n")
        for i, ln in enumerate(L):
            if re.search(pattern, ln):
                print(f"===== {f}:{i+1}")
                print("\n".join(L[max(0, i - ctx): i + ctx]))
                print()

print("########## INFERENCE-SIDE ##########")
try:
    from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
    print(inspect.getsource(OmniGen2Pipeline._apply_chat_template))
except Exception as e:
    print("import failed:", type(e).__name__, e)
    show(r"_apply_chat_template")

print("########## TRAINING-SIDE ##########")
show(r"use_chat_template")

########## INFERENCE-SIDE ##########


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


    def _apply_chat_template(self, prompt: str):
        prompt = [
            {
                "role": "system",
                "content": "You are a helpful assistant that generates high-quality images based on user instructions.",
            },
            {"role": "user", "content": prompt},
        ]
        prompt = self.processor.tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=False)
        return prompt

########## TRAINING-SIDE ##########
===== /content/OmniGen2/omnigen2/dataset/omnigen2_train_dataset.py:24
from torchvision import transforms

from datasets import load_dataset, concatenate_datasets

from ..pipelines.omnigen2.pipeline_omnigen2 import OmniGen2ImageProcessor

class OmniGen2TrainDataset(torch.utils.data.Dataset):
    SYSTEM_PROMPT = "You are a helpful assistant that generates high-quality images based on user instructions."
    SYSTEM_PROMPT_DROP = "You are a helpful assistant that generates images."

    def __init__(
        self,

/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:31: UserWarning: Cannot import triton, install triton to use fused RMSNorm for better performance
  warnings.warn("Cannot import triton, install triton to use fused RMSNorm for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


## 5. Callable templates
Builds a callable for each path without loading model weights. If the inference method needs
pipeline state it is reconstructed from the printed source.

In [5]:
templates = {}

# inference path: call the unbound method against a shim carrying only the processor
try:
    from transformers import AutoProcessor
    _proc = AutoProcessor.from_pretrained("OmniGen2/OmniGen2", subfolder="processor")
    class _Shim: pass
    _sh = _Shim(); _sh.processor = _proc; _sh.tokenizer = tok
    _f = OmniGen2Pipeline._apply_chat_template
    _ = _f(_sh, "probe")
    templates["inference"] = lambda s: _f(_sh, s)
    print("inference template: callable")
except Exception as e:
    print("inference template not directly callable:", type(e).__name__, e)

# training path: the dataset's own apply_chat_template, called unbound against a shim carrying
# only the two attributes it touches. Not a reconstruction — this is the method the trainer runs.
try:
    from omnigen2.dataset.omnigen2_train_dataset import OmniGen2TrainDataset
    class _TShim: pass
    _ts = _TShim(); _ts.tokenizer = tok; _ts.use_chat_template = True
    _tf = OmniGen2TrainDataset.apply_chat_template
    _TRAIN_SYS = OmniGen2TrainDataset.SYSTEM_PROMPT
    _ = _tf(_ts, "probe", _TRAIN_SYS)
    templates["training"] = lambda s: _tf(_ts, s, _TRAIN_SYS)
    print(f"training template: callable | SYSTEM_PROMPT is {len(tok(_TRAIN_SYS).input_ids)} tokens")
except Exception as e:
    print("training template not directly callable:", type(e).__name__, e)

# processor default, as a reference point
templates["processor_default"] = lambda s: tok.apply_chat_template(
    [{"role": "user", "content": s}], tokenize=False, add_generation_prompt=False)

for name, f in templates.items():
    print(f"\n----- {name} -----")
    print(repr(f("INSTRUCTION_HERE")))

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

inference template: callable
training template: callable | SYSTEM_PROMPT is 15 tokens

----- inference -----
'<|im_start|>system\nYou are a helpful assistant that generates high-quality images based on user instructions.<|im_end|>\n<|im_start|>user\nINSTRUCTION_HERE<|im_end|>\n'

----- training -----
'<|im_start|>system\nYou are a helpful assistant that generates high-quality images based on user instructions.<|im_end|>\n<|im_start|>user\nINSTRUCTION_HERE<|im_end|>\n'

----- processor_default -----
'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nINSTRUCTION_HERE<|im_end|>\n'


## 6. Overhead on real rows
Tests whether the overhead is constant, as the 25 figure assumes.

In [6]:
def overhead_table(f, rows, label):
    raw = [len(x) for x in tok([r["instruction"] for r in rows]).input_ids]
    tpl = [len(x) for x in tok([f(r["instruction"]) for r in rows]).input_ids]
    d   = collections.Counter(b - a for a, b in zip(raw, tpl))
    print(f"{label}: n={len(rows)}  overhead distribution {dict(sorted(d.items()))}")
    print(f"   constant: {len(d) == 1}  |  max raw {max(raw)}  max templated {max(tpl)}")
    return raw, tpl, d

MEAS = {}
for name, f in templates.items():
    print(f"\n########## {name} ##########")
    MEAS[name] = {"train": overhead_table(f, used, "train")}
    for s in ("val", "test"):
        MEAS[name][s] = overhead_table(f, splits[s], s)


########## inference ##########
train: n=39591  overhead distribution {25: 39591}
   constant: True  |  max raw 263  max templated 288
val: n=98  overhead distribution {25: 98}
   constant: True  |  max raw 199  max templated 224
test: n=488  overhead distribution {25: 488}
   constant: True  |  max raw 220  max templated 245

########## training ##########
train: n=39591  overhead distribution {25: 39591}
   constant: True  |  max raw 263  max templated 288
val: n=98  overhead distribution {25: 98}
   constant: True  |  max raw 199  max templated 224
test: n=488  overhead distribution {25: 488}
   constant: True  |  max raw 220  max templated 245

########## processor_default ##########
train: n=39591  overhead distribution {16: 39591}
   constant: True  |  max raw 263  max templated 279
val: n=98  overhead distribution {16: 98}
   constant: True  |  max raw 199  max templated 215
test: n=488  overhead distribution {16: 488}
   constant: True  |  max raw 220  max templated 236


## 7. The four documented figures, recomputed
Documented: templated max 288, P=668 max stream 956, P=1,247 max stream 1,535, and 96.23%
(38,098 of 39,591) of training rows at or below 888 with P=668.

In [7]:
CAP = 888
DOC = {"templated_max": 288, "p668_max": 956, "p1247_max": 1535,
       "inside_pct": 96.23, "inside_n": 38098, "over_n": 1493}

for name in MEAS:
    raw_tr, tpl_tr, d = MEAS[name]["train"]
    ov  = d.most_common(1)[0][0]
    mx  = max(tpl_tr)
    ins = sum(1 for t in tpl_tr if 668 + t <= CAP)
    print(f"\n########## {name}  (modal overhead {ov}) ##########")
    print(f"  templated max          {mx:>6}   documented {DOC['templated_max']:>6}   {'OK' if mx == DOC['templated_max'] else 'DIFFERS'}")
    print(f"  P=668  max stream      {668 + mx:>6}   documented {DOC['p668_max']:>6}   {'OK' if 668 + mx == DOC['p668_max'] else 'DIFFERS'}")
    print(f"  P=1247 max stream      {1247 + mx:>6}   documented {DOC['p1247_max']:>6}   {'OK' if 1247 + mx == DOC['p1247_max'] else 'DIFFERS'}")
    print(f"  rows inside 888        {ins:>6}   documented {DOC['inside_n']:>6}   {'OK' if ins == DOC['inside_n'] else 'DIFFERS'}")
    print(f"  rows over 888          {len(tpl_tr) - ins:>6}   documented {DOC['over_n']:>6}   {'OK' if len(tpl_tr) - ins == DOC['over_n'] else 'DIFFERS'}")
    print(f"  pct inside 888         {100 * ins / len(tpl_tr):>6.2f}   documented {DOC['inside_pct']:>6}")
    for s in ("test", "val"):
        _, tpl, _ = MEAS[name][s]
        print(f"  {s:<4} max stream at P=668 {668 + max(tpl):>5}   rows over 888 {sum(1 for t in tpl if 668 + t > CAP):>3} / {len(tpl)}")


########## inference  (modal overhead 25) ##########
  templated max             288   documented    288   OK
  P=668  max stream         956   documented    956   OK
  P=1247 max stream        1535   documented   1535   OK
  rows inside 888         38098   documented  38098   OK
  rows over 888            1493   documented   1493   OK
  pct inside 888          96.23   documented  96.23
  test max stream at P=668   913   rows over 888  14 / 488
  val  max stream at P=668   892   rows over 888   2 / 98

########## training  (modal overhead 25) ##########
  templated max             288   documented    288   OK
  P=668  max stream         956   documented    956   OK
  P=1247 max stream        1535   documented   1535   OK
  rows inside 888         38098   documented  38098   OK
  rows over 888            1493   documented   1493   OK
  pct inside 888          96.23   documented  96.23
  test max stream at P=668   913   rows over 888  14 / 488
  val  max stream at P=668   892   rows ove

## 8. Verdict

In [8]:
print("commit pinned      :", head == COMMIT, head)
for name in MEAS:
    d = MEAS[name]["train"][2]
    print(f"{name:<20} modal overhead {d.most_common(1)[0][0]}, constant {len(d) == 1}, distribution {dict(sorted(d.items()))}")
print()
if "training" in templates and "inference" in templates:
    _rows = [r["instruction"] for r in used] + [r["instruction"] for r in splits["val"]] \
            + [r["instruction"] for r in splits["test"]]
    _same = all(templates["training"](s) == templates["inference"](s) for s in _rows)
    print(f"training and inference render identical text on all {len(_rows)} rows "
          f"({len(used)} trained-on + {len(splits['val'])} val + {len(splits['test'])} test): {_same}")
    assert _same, "training and inference chat templates diverge"
    _ov = MEAS["training"]["train"][2]
    assert set(_ov) == {25}, f"training overhead is not constant 25: {dict(_ov)}"
    print("training-side overhead is a constant 25 -> the documented figures are measured, not inferred")
else:
    print("training callable unavailable; the overhead figures rest on source inspection only")
print("Section 7 must report OK on all five lines for the inference and training paths. The")
print("processor_default rows are the reference counterexample and are expected to report DIFFERS,")
print("since that path omits the OmniGen2 system prompt. A DIFFERS on inference or training would")
print("mean the four figures need reissuing; no generated artifact depends on them.")

commit pinned      : True 18e6f9d5271b517fcb32e999f10df943ae9b8f20
inference            modal overhead 25, constant True, distribution {25: 39591}
training             modal overhead 25, constant True, distribution {25: 39591}
processor_default    modal overhead 16, constant True, distribution {16: 39591}

training and inference render identical text on all 40177 rows (39591 trained-on + 98 val + 488 test): True
training-side overhead is a constant 25 -> the documented figures are measured, not inferred
Section 7 must report OK on all five lines for the inference and training paths. The
processor_default rows are the reference counterexample and are expected to report DIFFERS,
since that path omits the OmniGen2 system prompt. A DIFFERS on inference or training would
mean the four figures need reissuing; no generated artifact depends on them.
